# Pharos 04 - Grounded assistant (RAG)

The assistant retrieves from a curated knowledge base with TF-IDF and answers only from what it finds. Generation uses Gemini when a key is set and a local extractive fallback otherwise. Guardrails refuse personal dosing and diagnosis.

In [1]:
import sys, json
sys.path.insert(0,'../ml')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from domain import CATEGORIES, OPERATIONS

docs=[]
for atc,m in CATEGORIES.items():
    docs.append((f'{m["name"]} ({atc})', f"{m['name']} {m['group']} drugs {', '.join(m['examples'])}. {m['uses']} {m['seasonality']} {m['storage']} {m['cautions']}"))
for op in OPERATIONS: docs.append((op['title'], op['text']))
titles=[d[0] for d in docs]; corpus=[d[1] for d in docs]
vec=TfidfVectorizer(stop_words='english', ngram_range=(1,2), sublinear_tf=True)
M=vec.fit_transform(corpus); print(len(docs),'docs,',len(vec.vocabulary_),'terms')

13 docs, 709 terms


### Retrieval quality

In [2]:
def retrieve(q,k=3):
    sims=cosine_similarity(vec.transform([q]),M)[0]
    return [(titles[i], round(float(sims[i]),3)) for i in sims.argsort()[::-1][:k] if sims[i]>0.06]
for q in ['What is ibuprofen used for?','When does antihistamine demand peak?',
          'How should inhalers be stored?','How do reorder points work?']:
    print(q); [print('   ', s, t) for t,s in retrieve(q)]; print()

What is ibuprofen used for?
    0.151 Propionic acid NSAIDs (M01AE)
    0.146 Expiry management
    0.064 Salicylic acid analgesics (N02BA)

When does antihistamine demand peak?
    0.098 Systemic antihistamines (R06)

How should inhalers be stored?
    0.181 Airway disease drugs (R03)

How do reorder points work?
    0.231 How reorder points work



### Guardrails and language detection
Personal dosing and diagnosis requests are declined before any retrieval. Language is detected so the reply and the safety message match the user.

In [3]:
import re
UNSAFE=re.compile(r'how much|what dose|should i take|can i take|overdos|interact', re.I)
def detect(t):
    if re.search(r'[\u0600-\u06ff]', t): return 'ar'
    low=' '+t.lower()+' '
    for lang,ws in {'fr':[' le ',' la ',' pour ',' avec '],'es':[' el ',' para ',' con ']}.items():
        if sum(w in low for w in ws)>=2: return lang
    return 'en'
for q in ['How much paracetamol should I take?','Quel est l usage du paracetamol avec le repas','what is cetirizine']:
    print(f'{q!r:55s} unsafe={bool(UNSAFE.search(q))} lang={detect(q)}')

'How much paracetamol should I take?'                   unsafe=True lang=en
'Quel est l usage du paracetamol avec le repas'         unsafe=False lang=fr
'what is cetirizine'                                    unsafe=False lang=en


**Design.** Retrieval keeps answers grounded and auditable: every reply cites the documents it used and their similarity scores. The same index is exported to the browser so the public demo answers with no backend, while the API path adds fluent multilingual generation through Gemini. Guardrails sit in front of both paths, so the refusal behaviour does not depend on the language model.